# Reproducing Rajebhosale et al. (2025) from dandiset 001545

**Paper.** Rajebhosale P, Jiang L, Ressa HJ, Johnson KR, Desai NS, Jone A, Role LW, Talmage DA
(2025). Diversification of dentate gyrus granule cell subtypes is regulated by Nrg1 nuclear
back-signaling. *Life Science Alliance* 8(7): e202403169.
[doi:10.26508/lsa.202403169](https://doi.org/10.26508/lsa.202403169). The claims file was written
from the version of record in Europe PMC (PMC12032840), with its supplementary figure legends and
the Key Resources Table.

**Dandiset.** [DANDI:001545](https://dandiarchive.org/dandiset/001545/0.250809.1653), version
0.250809.1653 (doi:10.48324/dandi.001545/0.250809.1653). The paper cites no dandiset version (its
data statement points to GEO and GitHub only). This is the only published version; the draft
holds the same 48 assets (same asset ids), so the most recent published version is pinned.

**What the dandiset holds.** The whole-cell current clamp recordings of Fig 1J to M, Fig 2I to M
and Fig S2C to G: 48 NWB files, one cell each, from 5 WT and 5 V321L mice, each with the voltage
responses to a family of 500 ms current steps. Each file says the genotype (subject id VV or LL)
and the cell type (semilunar or not). It holds no soma positions, no morphology, and none of the
Golgi, RNAScope or snMultiome data, so 23 claims on those are blocked as data missing. The
snMultiome claims (`data: elsewhere`, GEO GSE280167) are blocked: they need the paper's R pipeline
and its curated nuclei and cluster labels, which GEO does not hold (see the data check).

**Method details.** The feature definitions are those of the claims file's opening comment (Kim
et al 2024 as the Methods' reference, and the paper's code, https://github.com/nsdesai/patch_clamp_analysis
at commit 5d756c2, where Kim et al are silent); each parameter and its source is in the docstring
of `nrg1_ephys.py`.

| Claim | Data | Status |
|---|---|---|
| fig1j-sgc-hyperpolarized-rmp | dandiset | tested |
| fig1k-sgc-narrower-spike | dandiset | tested |
| fig1l-sgc-faster-downstroke | dandiset | tested |
| fig1m-phase-plots | dandiset | tested |
| figs1-remaining-properties-no-difference | dandiset | tested (cell-type half only) |
| fig2i-v321l-rmp-not-different | dandiset | tested |
| fig2j-v321l-spike-width-not-different | dandiset | tested |
| fig2k-v321l-downstroke-not-different | dandiset | tested |
| fig2m-ephys-lda-wt-gc-apart | dandiset | tested, readings disagree |
| figs2d-ephys-kmeans-v321l-like-wt-sgc | dandiset | tested |
| figs2e-pca-separation-lost | dandiset | tested |
| figs2g-lda-classifier-accuracy | dandiset | tested |
| figs2f-centroid-separation-reduced | dandiset | tested |
| fig1f, fig1g, fig1h, fig1i, figs1a-f, figs1g-l, fig2c, fig2d, fig2ef, fig2g, figs2b, v321l-multiple-regression (soma position and morphology) | dandiset | blocked: data missing |
| fig2a (Golgi), fig3b, fig4a, fig4a-prime, fig4d, fig6a, fig6b, figs9a, figs9b, figs9d, fig7a (RNAScope) | dandiset | blocked: data missing |
| fig3d, figs3b, figs3c, figs3d, figs5a, figs5bc, fig4b, fig4b2, figs6, fig5b, figs7, trajectory-geometry, tables4, figs10d, fig7bc (snRNA-Seq) | elsewhere (GEO) | blocked: not runnable |
| fig5d, fig5f, figs8a, figs8b (snATAC-Seq) | elsewhere (GEO) | blocked: not runnable |
| figs9c-embryonic-sgc-genes (Allen ISH images) | elsewhere | blocked: other |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from statsmodels.stats.diagnostic import lilliefors

import nrg1_ephys as ne

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
cells = ne.load_all()
print(f"loaded {len(cells)} files from dandiset {ne.DANDISET_ID} version {ne.DANDISET_VERSION} in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
info = pd.DataFrame([{k: c[k] for k in ("identifier", "subject", "genotype", "cell_type", "age", "sex",
                                         "subject_description", "session_start")} | {"sweeps": len(c["v"])}
                     for c in cells])
print("Cells by genotype and cell type (genotype from the subject id, VV or LL; cell type from the session description):")
print(pd.crosstab(info.genotype, info.cell_type, margins=True).to_string())
print("\nPaper: WT 14 GC and 9 SGC in the electrophysiology (Fig 1J to L), V321L 10 GC and 15 SGC (Fig 2B, 2I to K).")
print("\nCells per mouse:")
print(pd.crosstab([info.genotype, info.subject], info.cell_type).to_string())
print("\nAges:", info.groupby("subject").age.first().to_dict())
print("Sexes:", info.groupby("subject").sex.first().to_dict())
print("\nSession descriptions:", sorted(set(c["session_description"] for c in cells)))
print("Subject descriptions:", sorted(set(c["subject_description"] for c in cells)))
print("Surgery field:", sorted(set(c["surgery"] for c in cells)))
print("Stimulus field:", sorted(set(c["stimulus_text"] for c in cells)))
print("Electrode cell_id values:", sorted(set(c["cell_id"] for c in cells)))

The subject ids name the genotype as VV or LL, and the `surgery` field says "wildtype (VV) or
neuregulin mutants (LL) containing a valine-to-leucine mutation"; the session descriptions agree
("wildtype" for every VV cell, "valine-to-leucine mutant" for every LL cell). VV is therefore taken
as WT and LL as V321L. The counts agree with this: the WT files hold 13 GCs and 9 SGCs from 5 mice
(the paper: 14 and 9 from 5 mice), the V321L files 12 GCs and 14 SGCs from 5 mice (the paper: 10
and 15 from 5 mice). So one WT GC is missing, and the V321L cell types differ from the paper's by
two GCs and one SGC. Nothing in the files says which cells the paper left out or relabelled.

In [ ]:
rows = []
for c in cells:
    start, stop, steps = ne.step_window(c)
    i = c["i"]
    changes = sorted(set(int(x) for s in i for x in np.nonzero(np.diff(s))[0]))
    rows.append(dict(cell=c["identifier"], genotype=c["genotype"], rate_hz=c["rate"], samples=c["v"].shape[1],
                     step_start_ms=start / c["rate"] * 1000, step_ms=(stop - start + 1) / c["rate"] * 1000,
                     stim_changes=str(changes), min_step=steps.min(), max_step=steps.max(),
                     n_neg60=int((steps == -60).sum()), n_neg20=int((steps == -20).sum()),
                     v_min=c["v"].min(), v_max=c["v"].max(), bias=c["bias_current"],
                     v_unit=c["voltage_unit"], i_unit=c["stim_unit"]))
chk = pd.DataFrame(rows)
print("Sampling rates (Hz):", chk.rate_hz.unique(), "; samples per sweep:", chk.samples.unique())
print("Stimulus change points (sample indices) in all files:", chk.stim_changes.unique())
print("Step onset (ms):", chk.step_start_ms.unique(), "; analysed step length (ms):", chk.step_ms.unique())
print("Stored stimulus values: min step", chk.min_step.min(), "max step", chk.max_step.max(),
      "; the unit attribute says", chk.i_unit.unique(), "with conversion 1")
print("Largest step per cell:", chk.max_step.value_counts().to_dict())
print("Cells with a -60 step:", (chk.n_neg60 > 0).sum(), "; with a -20 step:", (chk.n_neg20 > 0).sum())
print("Voltage unit attribute:", chk.v_unit.unique(), "; values after conversion to mV range from",
      round(chk.v_min.min(), 1), "to", round(chk.v_max.max(), 1))
print("bias_current attribute in every sweep (A):", chk.bias.unique())

Every sweep is 2 s at 10 kHz. The stimulus holds a step from 231.2 ms to 731.2 ms (500 ms) and a
-20 pA test pulse from 1731.2 to 1781.2 ms; the step amplitude is read at the end of the 500 ms
window. The stimulus values are -60 to 300, so they are picoamperes, while the files label them
amperes with a conversion of 1: a metadata error. The voltages are labelled volts and range from
about -0.1 to +0.05, which is plausible in volts. The general stimulus field says the steps run
from -60 to 200 pA; some cells were stepped to 250 or 300 pA.

In [ ]:
# Do the voltage and the stimulus share a clock? The response to the -60 pA step against the
# same measure with the stimulus shifted by 100 ms either way.
def deflection(c, shift_ms=0.0):
    start, stop, steps = ne.step_window(c)
    dt = 1000 / c["rate"]
    k = int(np.where(steps == -60)[0][0])
    s = start + int(round(shift_ms / dt))
    v = c["v"][k]
    return v[s + int(round(20 / dt)): s + int(round(60 / dt))].mean() - v[s - int(round(40 / dt)): s - int(round(1 / dt))].mean()

d = pd.DataFrame({f"shift {s:+.0f} ms": [deflection(c, s) for c in cells] for s in (-100, 0, 100)})
print("Voltage 20 to 60 ms after onset minus the 40 ms before it, -60 pA step (mV), over 48 cells:")
print(d.describe().loc[["mean", "min", "max"]].round(2).to_string())
lat = []
for c in cells:
    start, stop, steps = ne.step_window(c)
    k = int(np.where(steps == -60)[0][0])
    v = c["v"][k]
    base = v[start - 50: start].mean()
    lat.append((np.argmax(v[start - 50: start + 100] < base - 1.0) - 50) / c["rate"] * 1000)
print("Delay from step onset to a 1 mV hyperpolarization (ms): median", np.median(lat), "range", min(lat), max(lat))

# recordings deposited twice: identical sweeps across files
import hashlib, collections
hashes = collections.Counter(hashlib.md5(s.tobytes()).hexdigest() for c in cells for s in c["v"])
print("Sweeps:", sum(hashes.values()), "; sweeps whose data appear more than once:", sum(n for n in hashes.values() if n > 1))
same_start = info.groupby("session_start").size()
print("Session start times shared by more than one file:", int((same_start > 1).sum()))

The -60 pA step hyperpolarizes every cell from its onset (by 1 mV within 1.1 to 3.2 ms; mean
deflection -13.4 mV over 20 to 60 ms), and the same measure with the window shifted by 100 ms is near zero, so the stimulus and the voltage
share one clock. No sweep is deposited twice and no two files share a start time, so each file
is counted as one cell.

### Outside data: what GEO GSE280167 holds

The claims marked `data: elsewhere` use the paper's snMultiome data, deposited in GEO as
GSE280167. This cell reads the sample records and the barcode lists only, to say what is there.

In [ ]:
import io, requests
rows = []
for gsm in range(8590675, 8590691):
    txt = requests.get("https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi",
                       params=dict(acc=f"GSM{gsm}", targ="self", form="text", view="brief"), timeout=60).text
    fields = {}
    for line in txt.splitlines():
        if " = " in line:
            k, v = line.split(" = ", 1)
            fields.setdefault(k, []).append(v)
    ch = dict(x.split(": ", 1) for x in fields.get("!Sample_characteristics_ch1", []) if ": " in x)
    files = [f.rsplit("/", 1)[-1] for f in fields.get("!Sample_supplementary_file_1", []) + fields.get("!Sample_supplementary_file_2", []) + fields.get("!Sample_supplementary_file_3", [])]
    rows.append(dict(gsm=f"GSM{gsm}", title=fields["!Sample_title"][0], genotype=ch.get("genotype"),
                     treatment=ch.get("treatment"), sex=ch.get("Sex"), files=", ".join(f.split("_", 1)[1] for f in files)))
geo = pd.DataFrame(rows)
print(geo.to_string(index=False))
counts = {}
for r in rows[:8]:
    name = r["title"].replace("_DG_GEX", "").replace("_GEX", "")
    url = f"https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8590nnn/{r['gsm']}/suppl/{r['gsm']}_{name}_barcodes.tsv.gz"
    counts[name] = len(pd.read_csv(url, header=None))
print("\nBarcodes per expression sample:", counts)
feat_url = f"https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM8590nnn/{rows[0]['gsm']}/suppl/{rows[0]['gsm']}_VV_F1_features.tsv.gz"
print("Feature types in", rows[0]["gsm"], ":", pd.read_csv(feat_url, sep="\t", header=None)[2].value_counts().to_dict())

GEO confirms that VV is the wild type and LL the homozygous V321L mutant. It holds, per mouse,
the raw feature-barcode matrix of the multiome run (342,506 to 501,480 barcodes each, so every
barcode Cell Ranger saw, not the called nuclei; 32,285 genes and 68,588 peaks), and for ATAC only a
smoothed track of cut sites per sample. The per-nucleus fragment files the Methods import into
Signac, the nuclei kept after the code's filtering and doublet removal, and the cluster labels
(0, 1, 5, 18 and the others) that every snRNA-Seq criterion uses are not deposited.

In [ ]:
started = time.time()
feat = ne.feature_table(cells)
print(f"features computed in {time.time() - started:.0f} s")
show = ["rmp", "rheobase", "threshold", "spike_width", "spike_amplitude", "upstroke", "downstroke", "ud_ratio",
        "sag", "sag_ratio", "input_resistance", "input_resistance_code", "tau", "spike_latency", "ahp_amplitude",
        "ahp_latency", "ahp_width", "max_rate", "fi_slope", "adaptation_index", "cv", "spike_peak", "ahp_max",
        "threshold_code", "spike_width_code", "downstroke_codewin"]
print("Range of every feature over the 48 cells (units: mV, pA, ms, mV/ms, MOhm, Hz, Hz/pA):")
print(feat[show].describe().T[["count", "mean", "min", "max"]].round(2).to_string())
print("\nCells with a missing feature:")
print(feat[feat[ne.FEATURES_21].isna().any(axis=1)][["cell", "genotype", "cell_type"] + [f for f in ne.FEATURES_21 if feat[f].isna().any()]].to_string())

The ranges are those of dentate granule cells recorded in whole-cell current clamp: baselines of
-89 to -68 mV, rheobases of 30 to 210 pA, spike widths of about 1 to 3 ms and input resistances
of about 100 to 850 MOhm. The general stimulus field says that "neurons were kept near -65 mV
with a constant current", but the baseline before the steps is -68 to -89 mV in every cell, and
the `bias_current` attribute is 10 pA in every sweep of every file; the files do not say what
holding current, if any, was applied, so the baseline is used as the RMP, as the paper's code
does. The AHP width is missing where the code's half-level crossing is not found.

In [ ]:
def groups(df, feature, genotype):
    gc = df[(df.genotype == genotype) & (df.cell_type == "GC")][feature].dropna()
    sgc = df[(df.genotype == genotype) & (df.cell_type == "SGC")][feature].dropna()
    return gc, sgc

def mw(df, feature, genotype):
    gc, sgc = groups(df, feature, genotype)
    r = stats.mannwhitneyu(gc, sgc, alternative="two-sided")
    u_small = min(r.statistic, len(gc) * len(sgc) - r.statistic)
    return dict(feature=feature, genotype=genotype, n_gc=len(gc), n_sgc=len(sgc), gc_median=gc.median(),
                sgc_median=sgc.median(), U=u_small, U_gc=r.statistic, p=r.pvalue,
                welch_p=stats.ttest_ind(gc, sgc, equal_var=False).pvalue)

def welch(df, feature, genotype):
    gc, sgc = groups(df, feature, genotype)
    r = stats.ttest_ind(gc, sgc, equal_var=False)
    return dict(feature=feature, genotype=genotype, n_gc=len(gc), n_sgc=len(sgc), gc_mean=gc.mean(),
                sgc_mean=sgc.mean(), diff=gc.mean() - sgc.mean(), t=r.statistic, df=r.df, p=r.pvalue,
                mw_p=stats.mannwhitneyu(gc, sgc, alternative="two-sided").pvalue)

def by_mouse(df, feature, genotype):
    sub = df[df.genotype == genotype]
    return sub.groupby(["subject", "cell_type"])[feature].agg(["size", "median"]).unstack().round(2)

## fig1j-sgc-hyperpolarized-rmp

> In WT mice SGCs have a more hyperpolarized resting membrane potential than GCs.

Criterion: RMP per cell as the code defines it (baseline before step onset); two-tailed
Mann-Whitney test of WT SGCs against WT GCs; reproduced when p < 0.05 with SGCs more negative.
Paper: U = 26, P = 0.02, 14 GCs and 9 SGCs.

In [ ]:
r = mw(feat, "rmp", "WT")
print(pd.Series(r).to_string())
print("\nBy mouse (cells and median RMP, mV):")
print(by_mouse(feat, "rmp", "WT").to_string())
gc, sgc = groups(feat, "rmp", "WT")
loo = [stats.mannwhitneyu(gc.drop(i), sgc).pvalue for i in gc.index]
print(f"\nLeaving out one WT GC at a time: p from {min(loo):.3f} to {max(loo):.3f}")
fig, ax = plt.subplots(figsize=(3, 3))
for k, (g, x) in enumerate(((gc, 0), (sgc, 1))):
    ax.scatter(np.full(len(g), x) + np.linspace(-0.1, 0.1, len(g)), g, color=["silver", "gold"][k], edgecolor="k")
ax.set_xticks([0, 1], ["GC", "SGC"]); ax.set_ylabel("RMP (mV)"); ax.set_title("WT")
print("Plotted range:", round(min(gc.min(), sgc.min()), 1), "to", round(max(gc.max(), sgc.max()), 1), "mV; no points outside the axes")
plt.show()

**Result: not reproduced.** WT SGCs have a lower median RMP than WT GCs (-83.8 against -75.3 mV),
but the two-tailed Mann-Whitney test gives U = 32 and p = 0.083 over 13 GCs and 9 SGCs (paper:
U = 26, P = 0.02, 14 and 9). Welch's t test would give p = 0.044. The dandiset holds one WT GC
fewer than the paper; leaving out one WT GC at a time moves p between 0.043 and 0.127, so one cell
can move the test across 0.05. Within the two mice with both types and more than one SGC (VV317,
VV326) the medians differ by 1.6 to 2.1 mV.

## fig1k-sgc-narrower-spike

> In WT mice SGCs have narrower action potentials than GCs.

Criterion: width at half height between threshold (dV/dt 20 mV/ms, Kim et al) and peak of the
first spike at rheobase; the code's 10 mV/ms threshold is a variant. Two-tailed Mann-Whitney
test; reproduced when p < 0.05 with SGCs narrower. Paper: U = 24, P = 0.012.

In [ ]:
res = pd.DataFrame([mw(feat, "spike_width", "WT") | {"reading": "Kim et al, 20 mV/ms"},
                    mw(feat, "spike_width_code", "WT") | {"reading": "code, 10 mV/ms"}])
print(res.round(4).to_string())
print("\nBy mouse (cells and median width, ms):")
print(by_mouse(feat, "spike_width", "WT").to_string())
wt = feat[feat.genotype == "WT"]
other = wt[wt.subject != "NRG VV253"]
r2 = stats.mannwhitneyu(other[other.cell_type == "GC"].spike_width, other[other.cell_type == "SGC"].spike_width)
print(f"\nWithout mouse VV253 (4 GCs of 2.1 to 2.7 ms): GC median {other[other.cell_type == 'GC'].spike_width.median():.2f}, "
      f"SGC median {other[other.cell_type == 'SGC'].spike_width.median():.2f} ms, p = {r2.pvalue:.3f}")

**Result: not reproduced.** The WT SGC median spike width is 1.18 ms against 1.29 ms for WT GCs,
in the paper's direction, but the Mann-Whitney test gives U = 32, p = 0.082 (paper: U = 24,
P = 0.012); with the code's 10 mV/ms threshold U = 34, p = 0.109. The four GCs of mouse VV253 have
the widest spikes (median 2.24 ms) and its one SGC has 3.06 ms; without that mouse the medians
are 1.23 and 1.17 ms (p = 0.112).

## fig1l-sgc-faster-downstroke

> In WT mice SGCs have a faster action potential downstroke than GCs.

Criterion: the minimum dV/dt on the repolarizing phase of the first spike at rheobase (Kim et
al); the code's window (5 ms before the first peak to the end of the 500 ms window) is a second
reading. Two-tailed Mann-Whitney test; reproduced when p < 0.05 with SGCs more negative under
every reading. Paper: U = 31, P = 0.046.

In [ ]:
res = pd.DataFrame([mw(feat, "downstroke", "WT") | {"reading": "Kim et al, first spike"},
                    mw(feat, "downstroke_codewin", "WT") | {"reading": "code window"}])
print(res.round(4).to_string())
print("Cells whose downstroke differs between the readings:", int((feat.downstroke != feat.downstroke_codewin).sum()))
print("\nBy mouse (cells and median downstroke, mV/ms):")
print(by_mouse(feat, "downstroke", "WT").to_string())

**Result: not reproduced.** The WT SGC median downstroke is -61.95 mV/ms against -54.93 mV/ms for
WT GCs, in the paper's direction, but U = 44, p = 0.350 (paper: U = 31, P = 0.046). The two
readings give the same value in every cell (0 cells differ), as the first spike has the steepest
repolarization in every rheobase sweep.

## fig1m-phase-plots

> Phase plots of the action potential at rheobase show differences in waveform kinetics between
> WT GCs and SGCs.

Criterion: dV/dt against Vm minus threshold for the first spike at rheobase; judged as reproduced
when fig1k and fig1l are.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.2), sharex=True, sharey=True)
lims = []
for ax, g in zip(axes, ("WT", "V321L")):
    for c in cells:
        if c["genotype"] != g:
            continue
        pp = ne.phase_plot(c)
        ax.plot(pp[0], pp[1], color="gold" if c["cell_type"] == "SGC" else "gray", lw=0.8, alpha=0.8)
        lims.append((pp[0].min(), pp[0].max(), pp[1].min(), pp[1].max()))
    ax.set_title(f"{g}: GC grey, SGC gold"); ax.set_xlabel("Vm - Vth (mV)")
axes[0].set_ylabel("dV/dt (mV/ms)")
lims = np.array(lims)
print("Plotted ranges: Vm - Vth", lims[:, 0].min().round(1), "to", lims[:, 1].max().round(1),
      "mV; dV/dt", lims[:, 2].min().round(1), "to", lims[:, 3].max().round(1), "mV/ms; all inside the axes")
plt.show()
print("fig1k (Kim reading) p =", round(mw(feat, "spike_width", "WT")["p"], 4), "; fig1l (Kim reading) p =", round(mw(feat, "downstroke", "WT")["p"], 4))

**Result: not reproduced.** The criterion judges the phase plots through fig1k and fig1l, which
are not reproduced (p = 0.082 and 0.350). The plots are shown for both genotypes.

## figs1-remaining-properties-no-difference

> The remaining electrophysiological properties differ neither between WT GCs and SGCs nor with
> soma position.

Criterion: a claim of absence at p < 0.05. For each property other than RMP, spike width,
downstroke, rheobase, sag and the upstroke-downstroke ratio, WT GC against SGC by Welch's t test
when both groups pass the Shapiro-Wilk and Kolmogorov-Smirnov normality tests, and by Mann-Whitney
otherwise; and a regression on soma position. Readings (a) the 18 features of Fig S2C less the
five named and (b) the code's 21 outputs less the five. The dandiset holds no soma positions, so
only the cell-type half can be run here. The Kolmogorov-Smirnov normality test is taken as
Prism's, which uses the Lilliefors (Dallal-Wilkinson) correction.

In [ ]:
named = ["rmp", "spike_width", "downstroke", "rheobase", "sag", "ud_ratio"]
rows = []
for f in ne.FEATURES_21:
    if f in named:
        continue
    gc, sgc = groups(feat, f, "WT")
    normal = all(stats.shapiro(x).pvalue >= 0.05 and lilliefors(x, dist="norm")[1] >= 0.05 for x in (gc, sgc))
    t = stats.ttest_ind(gc, sgc, equal_var=False)
    u = stats.mannwhitneyu(gc, sgc, alternative="two-sided")
    rows.append(dict(feature=f, reading="a and b" if f in ne.FEATURES_18 else "b only", n_gc=len(gc), n_sgc=len(sgc),
                     gc_median=gc.median(), sgc_median=sgc.median(), normal=normal,
                     test="Welch" if normal else "Mann-Whitney", p=t.pvalue if normal else u.pvalue,
                     welch_p=t.pvalue, mw_p=u.pvalue,
                     cohen_d=(gc.mean() - sgc.mean()) / np.sqrt((gc.var() + sgc.var()) / 2)))
rem = pd.DataFrame(rows)
print(rem.round(4).to_string())
for rd, sub in (("a (18 features less 5)", rem[rem.reading == "a and b"]), ("b (21 outputs less 5)", rem)):
    print(f"Reading {rd}: {len(sub)} properties, {int((sub.p < 0.05).sum())} with p < 0.05:", sub[sub.p < 0.05].feature.tolist())

**Result: partially reproduced.** The cell-type half holds: none of the 13 remaining features of
reading (a), nor of the 16 of reading (b), differs between WT GCs and SGCs at p < 0.05 by the
paper's rule (smallest p 0.139, spike peak). The soma-position half cannot be run: the dandiset
holds no soma positions.

## fig2i-v321l-rmp-not-different

> In V321L mice GCs and SGCs do not differ in RMP; both lie near the WT SGC median.

Criterion: a claim of absence, judged on Welch's t test of V321L GC against SGC RMP at p < 0.05;
the difference and the group medians, beside the WT medians, are shown. Paper: t = 0.2081,
df = 18.86, P = 0.8374.

In [ ]:
r = welch(feat, "rmp", "V321L")
print(pd.Series(r).to_string())
med = feat.groupby(["genotype", "cell_type"]).rmp.median().round(2)
print("\nMedian RMP (mV):", med.to_dict())
print("\nBy mouse:")
print(by_mouse(feat, "rmp", "V321L").to_string())

**Result: reproduced.** V321L GCs and SGCs do not differ in RMP (Welch t = 0.54, df = 23.7,
p = 0.595; difference of means 1.2 mV; paper: t = 0.2081, P = 0.8374). The medians are -80.1 mV
(V321L GC) and -82.9 mV (V321L SGC), against -83.8 mV for WT SGCs and -75.3 mV for WT GCs, so the
V321L GC median is nearer the WT SGC median (3.7 mV) than the WT GC median (4.8 mV).

## fig2j-v321l-spike-width-not-different

> In V321L mice GCs and SGCs do not differ in spike width.

Criterion: a claim of absence, two-tailed Mann-Whitney test at p < 0.05, spike width as in fig1k;
the code's threshold is a variant. Paper: U = 55, P = 0.2776.

In [ ]:
res = pd.DataFrame([mw(feat, "spike_width", "V321L") | {"reading": "Kim et al, 20 mV/ms"},
                    mw(feat, "spike_width_code", "V321L") | {"reading": "code, 10 mV/ms"}])
print(res.round(4).to_string())
print("\nBy mouse:")
print(by_mouse(feat, "spike_width", "V321L").to_string())

**Result: reproduced.** V321L GC and SGC spike widths do not differ (medians 1.245 and 1.205 ms,
U = 59.5, p = 0.216; paper: U = 55, P = 0.2776); with the code's threshold p = 0.327.

## fig2k-v321l-downstroke-not-different

> In V321L mice GCs and SGCs do not differ in downstroke.

Criterion: a claim of absence, two-tailed Mann-Whitney test at p < 0.05, downstroke as in fig1l
(Kim et al); the code's window is a variant. Paper: U = 63.5, P = 0.5392.

In [ ]:
res = pd.DataFrame([mw(feat, "downstroke", "V321L") | {"reading": "Kim et al, first spike"},
                    mw(feat, "downstroke_codewin", "V321L") | {"reading": "code window"}])
print(res.round(4).to_string())
print("\nBy mouse:")
print(by_mouse(feat, "downstroke", "V321L").to_string())

**Result: reproduced.** V321L GC and SGC downstrokes do not differ (medians -59.05 and -57.53 mV/ms,
U = 80.5, p = 0.877; paper: U = 63.5, P = 0.5392); the code's window gives the same values.

## Multivariate analyses (Fig 2M, Fig S2D to G)

The 18 features of Fig S2C, z-scored over the cells analysed (`scale()`, Methods, LDA). Cells with
a missing feature are dropped (fig2m criterion); the same rule is used for the k-means, PCA and
classifier, whose criteria do not say. Two WT and two V321L cells lack the AHP width, which leaves
20 WT and 24 V321L cells; the paper's WT confusion matrix counts 21 cells and its V321L matrix 23.

In [ ]:
F = ne.FEATURES_18
mv = feat.dropna(subset=F).reset_index(drop=True)
mv["group"] = mv.genotype + " " + mv.cell_type
print("Cells kept:", mv.group.value_counts().to_dict(), "; dropped:", len(feat) - len(mv))

def zscore(x, ref=None):
    ref = x if ref is None else ref
    return (x - ref.mean()) / ref.std(ddof=1)

Z = zscore(mv[F]).values

## fig2m-ephys-lda-wt-gc-apart

> LDA of electrical properties separates WT GCs from all other groups (WT SGCs, V321L GCs, V321L
> SGCs) along LD1, driven mainly by spike width and downstroke.

Criterion: four-class LDA (genotype by cell type) on (a) the z-scored features and (b) their top
9 PCs. Reproduced when, under every reading, the WT GC mean LD1 lies on one side and the means of
the other three groups on the other, and spike width or downstroke has the largest absolute LD1
coefficient; blocked as methods_underspecified when the readings disagree. LDA here is
scikit-learn's (SVD solver, priors from the class sizes, as MASS lda); for reading (b) the LD1
coefficients are mapped back to the features through the PCA loadings.

In [ ]:
lda_res = {}
for reading in ("a", "b"):
    if reading == "a":
        X, load = Z, np.eye(len(F))
    else:
        pca = PCA(9).fit(Z)
        X, load = pca.transform(Z), pca.components_.T
    lda = LinearDiscriminantAnalysis().fit(X, mv.group)
    ld = lda.transform(X)
    means = pd.Series(ld[:, 0]).groupby(mv.group).mean()
    coef = pd.Series(load @ lda.scalings_[:, 0], index=F)
    top = coef.abs().sort_values(ascending=False)
    others = means.drop("WT GC")
    apart = means["WT GC"] > others.max() or means["WT GC"] < others.min()
    gap = min(abs(means["WT GC"] - others.max()), abs(means["WT GC"] - others.min()))
    lda_res[reading] = dict(apart=apart, top=top.index[0])
    print(f"Reading ({reading}): LD1 explains {lda.explained_variance_ratio_[0]:.2f} of the between-group variance")
    print("  mean LD1 by group:", means.round(2).to_dict())
    print("  extreme group on LD1:", means.idxmax() if abs(means.max()) > abs(means.min()) else means.idxmin(),
          "; WT GC apart from the other three:", apart, f"(gap to the nearest other mean {gap:.2f})")
    print("  largest |LD1 coefficients|:", (coef[top.index[:5]]).round(2).to_dict())
    print("  LD1 range by group:", pd.Series(ld[:, 0]).groupby(mv.group).agg(["min", "max"]).round(2).to_dict("index"))
print("\nReadings agree:", lda_res["a"]["apart"] == lda_res["b"]["apart"])

**Result: blocked (methods underspecified).** The two readings the criterion names disagree. On the
z-scored features (a) the WT GC mean LD1 (1.97) lies apart from the other three groups (-1.43 to
-0.33), and spike width has the largest LD1 coefficient (3.01, then downstroke -2.88), as the
paper says. On the top 9 PCs (b) the group apart on LD1 is WT SGC (1.67, the others -0.74 to
-0.26), and the largest coefficients are spike amplitude, AHP amplitude and sag (-0.48 to -0.43).

## figs2d-ephys-kmeans-v321l-like-wt-sgc

> k-means clustering (k = 3) of electrophysiological data places V321L GCs and SGCs in clusters in
> nearly the same proportions as WT SGCs, while WT GCs spread across all three clusters.

Criterion: k-means, k = 3, on (a) the z-scored features and (b) the first two PCs, 100 random
starts each. Reproduced when, under every reading, WT GCs occupy all three clusters with none
holding more than half of them, and WT SGCs, V321L GCs and V321L SGCs each have most cells in one
same cluster. The seed is fixed (a standard choice); the result for ten other seeds is shown.

In [ ]:
def kmeans_check(X, seed):
    lab = KMeans(3, n_init=100, random_state=seed).fit(X).labels_
    tab = pd.crosstab(mv.group, lab)
    wt_gc = tab.loc["WT GC"]
    spread = (wt_gc > 0).all() and wt_gc.max() <= wt_gc.sum() / 2
    modes = {g: tab.loc[g].idxmax() for g in ("WT SGC", "V321L GC", "V321L SGC")}
    majority = all(tab.loc[g, modes[g]] > tab.loc[g].sum() / 2 for g in modes)
    same = len(set(modes.values())) == 1
    return tab, spread, same, majority

X_pc2 = PCA(2).fit_transform(Z)
for reading, X in (("a", Z), ("b", X_pc2)):
    tab, spread, same, majority = kmeans_check(X, 0)
    print(f"Reading ({reading}), seed 0:")
    print(tab.to_string())
    print("  WT GCs in all three, none above half:", spread, "; other three groups share their largest cluster:", same,
          "; that cluster holds more than half of each:", majority)
    other = [kmeans_check(X, s)[1:] for s in range(1, 11)]
    print("  seeds 1 to 10, both conditions hold in", sum(a and b for a, b, _ in other), "of 10")

**Result: reproduced.** Under both readings WT GCs fall in all three clusters with at most half in
one (4, 6, 2 on the z-scored features; 6, 4, 2 on two PCs, of 12), and WT SGCs, V321L GCs and V321L
SGCs each have most of their cells in one same cluster (6 of 8, 7 of 10 and 10 of 14; 6 of 8, 6 of
10 and 10 of 14). Both conditions hold for ten other seeds under each reading.

## figs2e-pca-separation-lost

> PCA of the electrophysiological features separates WT GCs from WT SGCs on the first two
> components, while V321L GCs and SGCs overlap.

Criterion: PCA of the 18 z-scored features over all cells; 95% ellipses per group, (a) the normal
data ellipse (as ggplot2 stat_ellipse type "norm", radius from F(2, n - 1)) and (b) the
confidence ellipse of the group mean (Hotelling, F(2, n - 2)). Reproduced when, under every
reading, the WT GC and WT SGC centroids each lie outside the other group's ellipse and the V321L
centroids each lie inside the other's.

In [ ]:
pca2 = PCA(2).fit(Z)
P = pca2.transform(Z)
print("Variance explained by PC1 and PC2:", pca2.explained_variance_ratio_.round(3))

def inside(point, pts, kind):
    n = len(pts)
    mu, S = pts.mean(0), np.cov(pts.T)
    r2 = 2 * stats.f.ppf(0.95, 2, n - 1) if kind == "data" else 2 * (n - 1) / (n * (n - 2)) * stats.f.ppf(0.95, 2, n - 2)
    d2 = (point - mu) @ np.linalg.solve(S, point - mu)
    return d2, r2

rows = []
for g in ("WT", "V321L"):
    a = P[((mv.genotype == g) & (mv.cell_type == "GC")).values]
    b = P[((mv.genotype == g) & (mv.cell_type == "SGC")).values]
    for kind in ("data", "mean"):
        d1, r1 = inside(a.mean(0), b, kind)
        d2, r2 = inside(b.mean(0), a, kind)
        rows.append(dict(genotype=g, ellipse=kind, centroid_distance=np.linalg.norm(a.mean(0) - b.mean(0)),
                         gc_centroid_in_sgc_ellipse=d1 <= r1, d2_gc=d1, r2_sgc=r1,
                         sgc_centroid_in_gc_ellipse=d2 <= r2, d2_sgc=d2, r2_gc=r2))
ell = pd.DataFrame(rows)
print(ell.round(3).to_string())
print("\nPC scores range: PC1", P[:, 0].min().round(2), "to", P[:, 0].max().round(2), "; PC2", P[:, 1].min().round(2), "to", P[:, 1].max().round(2))
fig, ax = plt.subplots(figsize=(4, 4))
for grp, col, mk in (("WT GC", "k", "o"), ("WT SGC", "k", "^"), ("V321L GC", "teal", "o"), ("V321L SGC", "teal", "^")):
    m = (mv.group == grp).values
    ax.scatter(P[m, 0], P[m, 1], c=col, marker=mk, s=18, label=grp)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend(fontsize=7)
plt.show()

**Result: not reproduced.** The WT GC and WT SGC centroids each lie inside the other group's 95%
ellipse under both readings (squared distances 1.31 and 0.69 against radii 9.47 and 7.96 for the
data ellipse, 1.50 and 0.75 for the ellipse of the mean). The V321L centroids also lie inside each
other's ellipses, and they are closer together than the WT ones (0.86 against 2.07 PC units).

## figs2g-lda-classifier-accuracy

> An LDA classifier trained on WT electrophysiology classifies held-out WT cells with 78% accuracy
> (Cohen's kappa 0.57) but V321L cells with only 52% accuracy (kappa 0.08).

Criterion (Methods, LDA): the 18 features z-scored on WT, PCA on WT, top 9 PCs, LDA with
five-fold cross-validation; the WT PCA and LDA applied to the V321L data. Readings: V321L scaled
with its own mean and SD or with WT's; WT accuracy as the pooled out-of-fold accuracy or the mean
over folds; each over 100 random fold assignments (median and range). caret's folds are
stratified by class (createFolds), so stratified folds are used. The model applied to V321L is
the one fitted on all WT cells, as caret's finalModel is.

In [ ]:
wt = mv[mv.genotype == "WT"].reset_index(drop=True)
mu = mv[mv.genotype == "V321L"].reset_index(drop=True)
Zw = zscore(wt[F])
pca_w = PCA(9).fit(Zw.values)
Xw = pca_w.transform(Zw.values)
y = wt.cell_type.values
pooled, kappa, foldmean = [], [], []
for seed in range(100):
    pred = np.empty(len(y), dtype=object)
    accs = []
    for tr, te in StratifiedKFold(5, shuffle=True, random_state=seed).split(Xw, y):
        m = LinearDiscriminantAnalysis().fit(Xw[tr], y[tr])
        pred[te] = m.predict(Xw[te])
        accs.append(accuracy_score(y[te], pred[te]))
    pooled.append(accuracy_score(y, pred)); kappa.append(cohen_kappa_score(y, pred)); foldmean.append(np.mean(accs))
print(f"WT cells: {len(y)} ({(y == 'GC').sum()} GC, {(y == 'SGC').sum()} SGC); PCs kept 9, variance explained {pca_w.explained_variance_ratio_.sum():.2f}")
print(f"WT pooled out-of-fold accuracy: median {100 * np.median(pooled):.1f}% (range {100 * min(pooled):.1f} to {100 * max(pooled):.1f})")
print(f"WT mean accuracy over folds:    median {100 * np.median(foldmean):.1f}% (range {100 * min(foldmean):.1f} to {100 * max(foldmean):.1f})")
print(f"WT kappa (pooled): median {np.median(kappa):.2f} (range {min(kappa):.2f} to {max(kappa):.2f})")
print("Chance reference: always predicting the larger class gives", round(100 * max((y == 'GC').mean(), (y == 'SGC').mean()), 1), "%")
final = LinearDiscriminantAnalysis().fit(Xw, y)
v_res = {}
for scaling in ("own", "WT"):
    Zm = zscore(mu[F]) if scaling == "own" else zscore(mu[F], wt[F])
    Xm = pca_w.transform(Zm.values)
    pr = final.predict(Xm)
    v_res[scaling] = (Xm, pr)
    cm = confusion_matrix(mu.cell_type, pr, labels=["GC", "SGC"])
    print(f"\nV321L scaled with {scaling} mean and SD: accuracy {100 * accuracy_score(mu.cell_type, pr):.1f}%, "
          f"kappa {cohen_kappa_score(mu.cell_type, pr):.2f}; confusion (rows true GC, SGC; columns predicted GC, SGC): {cm.tolist()}")
print("Expected: WT 78 +- 7.8%, kappa 0.57 +- 0.1; V321L 52 +- 5.2%, kappa 0.08 +- 0.1")

**Result: not reproduced.** Over 100 random five-fold splits the WT classifier reaches a median of
55.0% (35.0 to 75.0%) pooled out-of-fold accuracy and kappa 0.12 (-0.33 to 0.49), below the 60%
of always predicting GC and far from 78% and 0.57. The mean over folds gives the same median. On
V321L cells it reaches 41.7% (kappa -0.11) with their own scaling and 45.8% (kappa -0.01) with the
WT scaling (paper: 52%, 0.08). WT accuracy exceeds V321L accuracy by 13.3 or 9.2 points, under the
15 points the criterion asks for a partial result.

## figs2f-centroid-separation-reduced

> In the WT-trained LDA, the distance between GC and SGC centroids along LD1 is much smaller in
> V321L than in WT, with both V321L centroids on the WT SGC side of the boundary.

Criterion: the classifier of figs2g; LD1 scores of WT cells and of V321L cells projected through
the WT PCA and LDA. Reproduced when the V321L GC-SGC centroid distance on LD1 is less than half
the WT distance and both V321L centroids lie on the WT SGC side of the LD1 decision boundary. The
boundary is the LD1 value where the classifier's decision function is zero (it includes the
class priors).

In [ ]:
s_w = final.transform(Xw)[:, 0]
w = final.coef_[0]
x0 = Xw.mean(0)
xb = x0 - final.decision_function(x0[None])[0] * w / (w @ w)
boundary = final.transform(xb[None])[0, 0]
cw = pd.Series(s_w).groupby(wt.cell_type).mean()
print(f"WT centroids on LD1: GC {cw['GC']:.2f}, SGC {cw['SGC']:.2f}; distance {abs(cw['GC'] - cw['SGC']):.2f}; boundary {boundary:.2f}")
sgc_side = np.sign(cw["SGC"] - boundary)
for scaling, (Xm, pr) in v_res.items():
    s_m = final.transform(Xm)[:, 0]
    cm_ = pd.Series(s_m).groupby(mu.cell_type).mean()
    dist = abs(cm_["GC"] - cm_["SGC"])
    both = all(np.sign(cm_[t] - boundary) == sgc_side for t in ("GC", "SGC"))
    print(f"V321L ({scaling} scaling): centroids GC {cm_['GC']:.2f}, SGC {cm_['SGC']:.2f}; distance {dist:.2f} "
          f"({dist / abs(cw['GC'] - cw['SGC']):.2f} of WT); both on the WT SGC side: {both}")
    print(f"   LD1 range {s_m.min():.2f} to {s_m.max():.2f}")
print(f"WT LD1 range {s_w.min():.2f} to {s_w.max():.2f}")
mid = (cw["GC"] + cw["SGC"]) / 2
for scaling, (Xm, pr) in v_res.items():
    cm_ = pd.Series(final.transform(Xm)[:, 0]).groupby(mu.cell_type).mean()
    print(f"Variant, boundary at the midpoint of the WT centroids ({mid:.2f}, equal priors): V321L ({scaling} scaling) "
          f"both centroids on the WT SGC side: {all(np.sign(cm_[t] - mid) == sgc_side for t in ('GC', 'SGC'))}")

**Result: partially reproduced.** The V321L GC-SGC distance on LD1 is 0.17 (own scaling) or 0.30
(WT scaling), 0.08 or 0.14 of the WT distance of 2.18, so the first condition holds. Both V321L
centroids lie on the WT GC side of the boundary (0.40), not the SGC side (-0.10 and 0.07, or -0.47
and -0.17), and also of the midpoint boundary (0.22). The WT classifier on which this rests
separates held-out WT cells no better than chance (figs2g).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| fig1j-sgc-hyperpolarized-rmp | dandiset | not reproduced |
| fig1k-sgc-narrower-spike | dandiset | not reproduced |
| fig1l-sgc-faster-downstroke | dandiset | not reproduced |
| fig1m-phase-plots | dandiset | not reproduced |
| figs1-remaining-properties-no-difference | dandiset | partially reproduced |
| fig2i-v321l-rmp-not-different | dandiset | reproduced |
| fig2j-v321l-spike-width-not-different | dandiset | reproduced |
| fig2k-v321l-downstroke-not-different | dandiset | reproduced |
| fig2m-ephys-lda-wt-gc-apart | dandiset | blocked: methods underspecified |
| figs2d-ephys-kmeans-v321l-like-wt-sgc | dandiset | reproduced |
| figs2e-pca-separation-lost | dandiset | not reproduced |
| figs2g-lda-classifier-accuracy | dandiset | not reproduced |
| figs2f-centroid-separation-reduced | dandiset | partially reproduced |
| 12 claims on soma position and morphology, 11 on Golgi and RNAScope | dandiset | blocked: data missing |
| 19 snMultiome claims | elsewhere (GEO) | blocked: not runnable |
| figs9c-embryonic-sgc-genes | elsewhere (Allen) | blocked: other |

**How the analysis differs from the paper's.** The features follow the claims file's definitions
(Kim et al, with the paper's MATLAB code where Kim et al are silent), reimplemented in Python;
the paper names twenty features and the claims file eighteen. Statistics are scipy's in place of
Prism's, and LDA, PCA and k-means are scikit-learn's in place of R's MASS, prcomp, caret and
kmeans. Cells with a missing feature (the AHP width, 4 cells) are left out of the multivariate
analyses. The WT tests use 13 GCs where the paper has 14, and the V321L tests 12 GCs and 14 SGCs
where the paper has 10 and 15.

**What was found about the dataset.** The files hold 22 WT cells (13 GC, 9 SGC) and 26 V321L
cells (12 GC, 14 SGC) from 5 mice each; the paper reports 14 and 9, and 10 and 15. The genotype is
coded VV and LL; the files' surgery field and GEO say VV is the wild type. Every stimulus series
stores picoamperes under the unit amperes. The electrode `cell_id` is "VP neuron" in every file,
though the cells are dentate granule cells. The stimulus field says cells were held near -65 mV
and stepped from -60 to 200 pA, while the baselines are -68 to -89 mV and 14 cells were stepped
to 250 or 300 pA. The soma positions and dendritic morphology that 12 claims need are not in the
dandiset. The WT GC-SGC differences of Fig 1J to L are in the paper's direction but not
significant here (p = 0.08 to 0.35), and the WT-trained classifier does not separate WT GCs from
SGCs better than chance.